[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GarretOS/python-ai-foundations/blob/main/projects/python-foundations-quiz/python_foundations_quiz.ipynb)


# 🐍 Python Foundations Quiz

This notebook teaches and demonstrates a small beginner-friendly quiz based on the Towards AI **Interactive Quiz App with Gradio** lesson.

## 🎯 Project Overview

The quiz uses Gradio to display nine easy Python questions, answer choices, and friendly feedback. The local `quiz_app.py` file is the primary application. This notebook is a supplementary learning companion.

## 🐍 Python Concepts

This project practices:

- Lists and dictionaries
- List comprehensions
- `enumerate()` for question lookup
- `None` for a missing question result
- Functions, imports, and return values
- `if` / `else` control flow
- The `with` statement
- `if __name__ == "__main__":`

## 📦 Modules and Quiz Data

A module is a Python file that can be imported by another file. In the real multi-file application, `quiz_app.py` imports the local module beside it with:

```python
from quiz_data import QUIZ_QUESTIONS
```

`quiz_data.py` is intentionally part of this project: it demonstrates modules and keeps the quiz data organized separately from the application. When this notebook is opened by itself from GitHub in Google Colab, sibling files are not automatically copied into the runtime. The next cell therefore contains a self-contained copy of the small dataset so the learning examples can run independently.

In [ ]:
QUIZ_QUESTIONS = [
    {
        "question": "Which function displays text on the screen?",
        "options": ["print()", "show()", "display_text()", "write_screen()"],
        "answer": "print()",
        "explanation": "print() displays text or other values in the console.",
    },
    {
        "question": "Which keyword is used to define a function?",
        "options": ["def", "function", "make", "fun"],
        "answer": "def",
        "explanation": "The def keyword starts a function definition in Python.",
    },
    {
        "question": "Which data structure stores key-value pairs?",
        "options": ["Dictionary", "List", "String", "Tuple"],
        "answer": "Dictionary",
        "explanation": "A dictionary stores values under named keys.",
    },
    {
        "question": "What does input() do?",
        "options": ["Reads text from the user", "Deletes a file", "Draws a picture", "Stops Python"],
        "answer": "Reads text from the user",
        "explanation": "input() pauses the program and reads text typed by the user.",
    },
    {
        "question": "Which loop can go through items in a list?",
        "options": ["for loop", "if loop", "define loop", "print loop"],
        "answer": "for loop",
        "explanation": "A for loop can repeat an action for each item in a list.",
    },
    {
        "question": "Which symbol starts a comment in Python?",
        "options": ["#", "//", "<!--", "comment:"],
        "answer": "#",
        "explanation": "Python ignores text after # on the same line as code.",
    },
    {
        "question": "Which value means that there is no value?",
        "options": ["None", "Empty", "Nothing", "False"],
        "answer": "None",
        "explanation": "None is Python's value for no value or a missing result.",
    },
    {
        "question": "Which brackets create a list?",
        "options": ["Square brackets []", "Curly braces {}", "Parentheses ()", "Angle brackets <>"],
        "answer": "Square brackets []",
        "explanation": "Square brackets create a list, such as [1, 2, 3].",
    },
    {
        "question": "What does len() tell you?",
        "options": ["How many items there are", "The largest number", "The data type", "The last item"],
        "answer": "How many items there are",
        "explanation": "len() returns the number of characters or items in a value.",
    },
]

print(f"Number of questions: {len(QUIZ_QUESTIONS)}")
print(QUIZ_QUESTIONS[0])

In [ ]:
!pip install -q "gradio>=5.0,<7.0"

## 🔎 List Comprehensions and `enumerate()`

A list comprehension creates a new list in a compact way. The application uses one to extract every question label. `enumerate()` supplies both an index and the dictionary at that position, which lets the app find the selected question without an unused index.

In [ ]:
question_texts = [question["question"] for question in QUIZ_QUESTIONS]

for index, question in enumerate(QUIZ_QUESTIONS):
    print(index, question["question"])

## 0️⃣ `None` and Question Lookup

`None` can represent a missing result. The application returns `None` when a selected question cannot be found, and the interface responds with an empty set of answer choices instead of crashing.

In [ ]:
def find_question_index(selected_question):
    for index, question in enumerate(QUIZ_QUESTIONS):
        if question["question"] == selected_question:
            return index
    return None

print(find_question_index(question_texts[0]))
print(find_question_index("This question is not in the quiz."))

## 📚 Gradio, `pip`, and Virtual Environments

Gradio is a third-party library, so it is listed in `requirements.txt` and installed with `pip`. A virtual environment keeps this project's packages separate from other Python projects. The command-line workflow is:

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
```

## 🧩 Gradio Components and Events

The app uses `with gr.Blocks() as demo:` to group its components. A `Dropdown` selects a question, a read-only `Textbox` displays it, a `Radio` component displays answer options, and a `Button` checks the answer. The dropdown's `.change()` event returns updated component configurations for the question and choices, resets the Radio value, and clears feedback. The button's `.click()` event displays feedback. Running `demo.launch()` starts the web app; in Google Colab, Gradio may provide a temporary `gradio.live` share URL that you can open in a browser. The link is temporary, not a permanent deployment.

In [ ]:
import gradio as gr

def show_question(selected_question):
    question_index = find_question_index(selected_question)
    if question_index is None:
        return (
            gr.Textbox(value=None),
            gr.Radio(choices=[], value=None),
            gr.Textbox(value=""),
        )
    question = QUIZ_QUESTIONS[question_index]
    return (
        gr.Textbox(value=question["question"]),
        gr.Radio(
            choices=question["options"],
            value=None,
        ),
        gr.Textbox(value=""),
    )

def check_answer(selected_question, selected_option):
    question_index = find_question_index(selected_question)
    if question_index is None or not selected_option:
        return "Please select a question and an answer first."
    question = QUIZ_QUESTIONS[question_index]
    if selected_option == question["answer"]:
        return "Correct! 🎉"
    else:
        return f"Not quite. {question['explanation']}"

with gr.Blocks() as demo:
    gr.Markdown("# Python Foundations Quiz")
    selector = gr.Dropdown(choices=question_texts, label="Select a question", value=None)
    display = gr.Textbox(label="Question", interactive=False)
    choices = gr.Radio(choices=[], label="Choose an answer")
    check_button = gr.Button("Check Answer")
    feedback = gr.Textbox(label="Feedback", interactive=False)
    selector.change(show_question, selector, [display, choices, feedback])
    check_button.click(check_answer, [selector, choices], feedback)

demo.launch()

## 🧪 Try It Yourself

Try these easy experiments without starting a Gradio server:

1. Change the list comprehension so it extracts every answer instead of every question.
2. Use `enumerate()` to print the questions starting at number 1.
3. Test the answer-checking function with one correct and one incorrect choice.

In [ ]:
answers = [question["answer"] for question in QUIZ_QUESTIONS]

for number, question in enumerate(QUIZ_QUESTIONS, start=1):
    print(f"{number}. {question['question']}")

first_question = QUIZ_QUESTIONS[0]["question"]
print(check_answer(first_question, QUIZ_QUESTIONS[0]["answer"]))
print(check_answer(first_question, QUIZ_QUESTIONS[0]["options"][1]))

## 📚 What I Learned

I practiced organizing quiz data in a local module, importing a constant, extracting values with a list comprehension, and using `enumerate()` for lookup. I also connected `pip`, virtual environments, `requirements.txt`, Gradio components, the `with` statement, event handlers, and the `if __name__ == "__main__":` guard.

## 📝 Notes

- The questions are intentionally easy so the Gradio application remains friendly to beginners.
- `quiz_app.py` is the primary application; this notebook is supplementary.
- In Google Colab, run the Gradio launch cell to use the quiz; it may provide a temporary `gradio.live` share URL for browser testing, not a permanent deployment.
- `quiz_app.py` remains the primary real application, and this notebook is a learning/testing companion.
- Gradio is a third-party dependency listed in `requirements.txt`.
- No live Hugging Face Spaces deployment is included.